# 05 — Visualize & Browse Database

Interactive browser for the Philippine Supreme Court cases SQLite database produced by Notebook 04.
Uses `ipywidgets` for dropdown selection and click-through case detail views.

## Setup & Database Connection

In [1]:
import sqlite3
import json
import textwrap
from pathlib import Path

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from dotenv import load_dotenv
import os

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 100)

In [2]:
# --- Prompt for database path (falls back to .env / default) ---
load_dotenv()
_default_db = os.getenv("DB_PATH", "./sc_decisions.db")

db_path_input = input(f"Enter SQLite database path [{_default_db}]: ").strip()
DB_PATH = Path(db_path_input) if db_path_input else Path(_default_db)

if not DB_PATH.exists():
    raise FileNotFoundError(f"Database not found: {DB_PATH.resolve()}")

conn = sqlite3.connect(str(DB_PATH))
conn.row_factory = sqlite3.Row
print(f"Connected to: {DB_PATH.resolve()}")

# --- Verify expected tables ---
EXPECTED_TABLES = {"cases", "volume_toc", "processing_log"}
tables = {r[0] for r in conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table'"
).fetchall()}

missing = EXPECTED_TABLES - tables
if missing:
    raise RuntimeError(f"Missing tables: {missing}. Run Notebook 04 first.")

print(f"Tables verified: {sorted(EXPECTED_TABLES)}")

Connected to: C:\Users\Gamers\GitHub\SC-Decisions\sc_decisions.db
Tables verified: ['cases', 'processing_log', 'volume_toc']


---
## Database Overview

In [3]:
# --- Row counts ---
counts = {}
for tbl in sorted(EXPECTED_TABLES):
    counts[tbl] = conn.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]

display(HTML("<h3>Row Counts</h3>"))
display(pd.DataFrame.from_dict(counts, orient="index", columns=["rows"]))

,rows
cases,36558
processing_log,860
volume_toc,78322


In [4]:
# --- Cases table schema ---
COLUMN_NOTES = {
    "id":                 "Auto-increment primary key",
    "source_filename":    "Volume PDF filename (volume identifier)",
    "gr_numbers":         "JSON array of G.R. / case numbers",
    "date_of_decision":   "Decision date as text",
    "division":           "En Banc / First Division / etc.",
    "ponente_name":       "Writing justice name",
    "ponente_title":      "Justice title (J. / C.J. / etc.)",
    "ponente_role":       "Role annotation if any",
    "parties":            "JSON — party names",
    "counsel":            "JSON — counsel for each party",
    "justices":           "JSON — participating justices",
    "syllabus":           "JSON — syllabus / headnotes",
    "main_decision_text": "Full decision text body",
    "separate_opinions":  "JSON — concurring / dissenting opinions",
    "footnotes":          "JSON — extracted footnotes",
    "pages":              "JSON — page range within volume",
    "parse_incomplete":   "1 if parsing flagged issues",
    "parse_errors":       "JSON — error messages during parse",
    "date_processed":     "Timestamp when record was written",
}

schema_rows = conn.execute("PRAGMA table_info(cases)").fetchall()
schema_df = pd.DataFrame(
    [(r["name"], r["type"], COLUMN_NOTES.get(r["name"], "")) for r in schema_rows],
    columns=["column", "type", "description"],
)
display(HTML("<h3>Cases Table Schema</h3>"))
display(schema_df)

,column,type,description
0,id,INTEGER,Auto-increment primary key
1,source_filename,TEXT,Volume PDF filename (volume identifier)
2,gr_numbers,TEXT,JSON array of G.R. / case numbers
3,date_of_decision,TEXT,Decision date as text
4,division,TEXT,En Banc / First Division / etc.
5,ponente_name,TEXT,Writing justice name
6,ponente_title,TEXT,Justice title (J. / C.J. / etc.)
7,ponente_role,TEXT,Role annotation if any
8,parties,TEXT,JSON — party names
9,counsel,TEXT,JSON — counsel for each party


In [5]:
# --- Sample rows ---
SAMPLE_COLS = [
    "id", "source_filename", "gr_numbers", "parties",
    "date_of_decision", "ponente_name", "parse_incomplete",
]

sample_df = pd.read_sql_query(
    f"SELECT {', '.join(SAMPLE_COLS)} FROM cases LIMIT 30", conn
)


def _truncate(val, maxlen=60):
    if not isinstance(val, str):
        return val
    return val[:maxlen] + "..." if len(val) > maxlen else val


display(HTML("<h3>Sample Cases (first 30 rows)</h3>"))
display(sample_df.map(_truncate))

,id,source_filename,gr_numbers,parties,date_of_decision,ponente_name,parse_incomplete
0,19928,Volume_121.pdf,"[""No. L-12752""]",[],"January 80, 1965",None,0
1,19929,Volume_121.pdf,"[""No. L-16392""]","[{""gr_number"": ""No. L-16392"", ""petitioners"": [""PpoPLE OF THE...",None,None,0
2,19930,Volume_121.pdf,"[""No. L~16463""]",[],"January 30, 1965",None,0
3,19931,Volume_121.pdf,"[""No. L-16485"", ""No-""]","[{""gr_number"": ""No. L-16485"", ""petitioners"": [""REPUBLIC OF T...","January 80, 1965",None,0
4,19932,Volume_121.pdf,[],[],None,None,0
5,19933,Volume_121.pdf,"[""No. L-17641""]",[],"January 80, 1965",None,0
6,19934,Volume_121.pdf,"[""No. L-17996"", ""No L-18056""]",[],"January 30, 1965",Regala,0
7,19935,Volume_121.pdf,"[""No. L-18387""]","[{""gr_number"": ""No. L-18387"", ""petitioners"": [""Cuua CHE""], ""...","January 30, 1965",None,0
8,19936,Volume_121.pdf,"[""No. L-19118""]","[{""gr_number"": ""No. L-19118"", ""petitioners"": [""(With Resolut...","January 30, 1965",None,0
9,19937,Volume_121.pdf,"[""No. L-19309""]",[],"January 30, 1965",None,0


---
## Interactive Volume Browser

Select a volume (source filename) from the dropdown to see its cases.  
Click a case row to view full structured data.

In [6]:
# ── helpers ────────────────────────────────────────────────────────────────

JSON_FIELDS = [
    "gr_numbers", "parties", "counsel", "justices",
    "syllabus", "separate_opinions", "footnotes", "pages", "parse_errors",
]


def _safe_json(val):
    """Try to parse a JSON string; return raw value on failure."""
    if not val:
        return None
    try:
        return json.loads(val)
    except (json.JSONDecodeError, TypeError):
        return val


def _format_json_html(obj, indent=0):
    """Render a parsed JSON value as indented HTML."""
    pad = "&nbsp;" * indent
    if obj is None:
        return f"{pad}<em>—</em>"
    if isinstance(obj, list):
        if not obj:
            return f"{pad}<em>(empty list)</em>"
        parts = []
        for i, item in enumerate(obj, 1):
            parts.append(f"{pad}{i}. {_format_json_html(item, indent + 2)}")
        return "<br>".join(parts)
    if isinstance(obj, dict):
        if not obj:
            return f"{pad}<em>(empty)</em>"
        parts = []
        for k, v in obj.items():
            parts.append(f"{pad}<b>{k}:</b> {_format_json_html(v, indent + 2)}")
        return "<br>".join(parts)
    text = str(obj)
    if len(text) > 500:
        text = text[:500] + " [...]"
    return f"{pad}{text}"


# ── query helpers ──────────────────────────────────────────────────────────

def get_volumes():
    rows = conn.execute(
        "SELECT DISTINCT source_filename FROM cases ORDER BY source_filename"
    ).fetchall()
    return [r[0] for r in rows]


def get_cases_for_volume(vol):
    return pd.read_sql_query(
        "SELECT id, gr_numbers, parties, date_of_decision, "
        "ponente_name, parse_incomplete FROM cases "
        "WHERE source_filename = ? ORDER BY id",
        conn, params=(vol,),
    )


def get_case_by_id(case_id):
    row = conn.execute("SELECT * FROM cases WHERE id = ?", (case_id,)).fetchone()
    return dict(row) if row else None


# ── widget layout ─────────────────────────────────────────────────────────

volumes = get_volumes()

volume_dropdown = widgets.Dropdown(
    options=[("-- select a volume --", None)] + [(v, v) for v in volumes],
    description="Volume:",
    style={"description_width": "60px"},
    layout=widgets.Layout(width="500px"),
)

output_area = widgets.Output(layout=widgets.Layout(
    border="1px solid #ccc", padding="12px", width="100%",
    max_height="700px", overflow_y="auto",
))


# ── render: case detail ───────────────────────────────────────────────────

def show_case_detail(case_id, vol):
    output_area.clear_output(wait=True)
    with output_area:
        case = get_case_by_id(case_id)
        if not case:
            display(HTML("<p>Case not found.</p>"))
            return

        # Back button
        back_btn = widgets.Button(
            description="\u2190 Back to volume list",
            button_style="info",
            layout=widgets.Layout(margin="0 0 12px 0"),
        )
        back_btn.on_click(lambda _: show_volume_cases(vol))
        display(back_btn)

        # Warning banner
        if case.get("parse_incomplete"):
            display(HTML(
                '<div style="background:#fff3cd;border:1px solid #ffc107;'
                'padding:8px;margin-bottom:10px;border-radius:4px;">'
                '<strong>Warning:</strong> This case is flagged as '
                '<code>parse_incomplete</code>.</div>'
            ))

        # Title line
        gr = _safe_json(case.get("gr_numbers")) or []
        gr_str = ", ".join(gr) if isinstance(gr, list) else str(gr)
        display(HTML(f"<h3>Case #{case['id']} &mdash; {gr_str}</h3>"))

        # Build detail sections
        DETAIL_ORDER = [
            ("Source file",       "source_filename", False),
            ("G.R. Numbers",      "gr_numbers",      True),
            ("Date of Decision",  "date_of_decision", False),
            ("Division",          "division",        False),
            ("Ponente",           "ponente_name",    False),
            ("Ponente Title",     "ponente_title",   False),
            ("Ponente Role",      "ponente_role",    False),
            ("Parties",           "parties",         True),
            ("Counsel",           "counsel",         True),
            ("Justices",          "justices",        True),
            ("Syllabus",          "syllabus",        True),
            ("Separate Opinions", "separate_opinions", True),
            ("Footnotes",         "footnotes",       True),
            ("Pages",             "pages",           True),
            ("Parse Errors",      "parse_errors",    True),
            ("Parse Incomplete",  "parse_incomplete", False),
            ("Date Processed",    "date_processed",  False),
        ]

        html_parts = ['<table style="border-collapse:collapse;width:100%">']
        for label, key, is_json in DETAIL_ORDER:
            val = case.get(key)
            if is_json:
                parsed = _safe_json(val)
                rendered = _format_json_html(parsed)
            else:
                rendered = str(val) if val is not None else "<em>—</em>"
            html_parts.append(
                f'<tr style="border-bottom:1px solid #eee;vertical-align:top">'
                f'<td style="padding:6px 12px 6px 0;font-weight:bold;'
                f'white-space:nowrap">{label}</td>'
                f'<td style="padding:6px 0">{rendered}</td></tr>'
            )
        html_parts.append("</table>")
        display(HTML("\n".join(html_parts)))

        # Main decision text (collapsible)
        body = case.get("main_decision_text") or ""
        if body:
            preview = body[:2000].replace("\n", "<br>")
            suffix = "..." if len(body) > 2000 else ""
            display(HTML(
                "<details style='margin-top:12px'>"
                "<summary><strong>Main Decision Text</strong> "
                f"({len(body):,} chars)</summary>"
                f"<div style='white-space:pre-wrap;font-size:0.9em;'"
                f"padding:8px'>{preview}{suffix}</div>"
                "</details>"
            ))


# ── render: volume case list ──────────────────────────────────────────────

def show_volume_cases(vol):
    output_area.clear_output(wait=True)
    with output_area:
        df = get_cases_for_volume(vol)
        n = len(df)
        incomplete = df["parse_incomplete"].sum()
        display(HTML(
            f"<h3>{vol}</h3>"
            f"<p><strong>{n}</strong> case(s)"
            f"{f' &mdash; <span style=color:orange>{incomplete} flagged incomplete</span>' if incomplete else ''}"
            "</p>"
        ))

        if n == 0:
            display(HTML("<p>No cases found for this volume.</p>"))
            return

        # Render each case as a clickable button row
        for _, row in df.iterrows():
            gr = _safe_json(row["gr_numbers"])
            gr_str = ", ".join(gr) if isinstance(gr, list) else str(gr or "—")
            parties = _safe_json(row["parties"])
            if isinstance(parties, list):
                title = " | ".join(str(p) for p in parties[:3])
            elif isinstance(parties, str):
                title = parties
            else:
                title = "—"
            title = title[:80] + "..." if len(title) > 80 else title

            flag = " [INCOMPLETE]" if row["parse_incomplete"] else ""
            label = f"{gr_str}  |  {row['date_of_decision'] or '—'}  |  {row['ponente_name'] or '—'}{flag}"

            btn = widgets.Button(
                description="",
                tooltip=title,
                layout=widgets.Layout(width="100%", height="auto"),
            )
            # Use HTML widget for richer display inside a clickable HBox
            btn_html = widgets.HTML(
                value=(
                    f"<div style='padding:4px 8px;cursor:pointer'>"
                    f"<strong>{gr_str}</strong> &mdash; {title}<br>"
                    f"<small>{row['date_of_decision'] or '—'} &bull; "
                    f"{row['ponente_name'] or '—'}"
                    f"{'&nbsp;<span style=color:orange>[INCOMPLETE]</span>' if row['parse_incomplete'] else ''}"
                    f"</small></div>"
                )
            )
            btn = widgets.Button(
                description=label[:70],
                button_style="warning" if row["parse_incomplete"] else "",
                layout=widgets.Layout(width="100%"),
            )
            case_id = int(row["id"])
            btn.on_click(lambda _, cid=case_id, v=vol: show_case_detail(cid, v))
            display(btn)


# ── dropdown handler ──────────────────────────────────────────────────────

def on_volume_change(change):
    vol = change["new"]
    if vol is None:
        output_area.clear_output()
        return
    show_volume_cases(vol)

volume_dropdown.observe(on_volume_change, names="value")

# ── display ───────────────────────────────────────────────────────────────

display(HTML(f"<p><strong>{len(volumes)}</strong> volumes available</p>"))
display(volume_dropdown)
display(output_area)

Dropdown(description='Volume:', layout=Layout(width='500px'), options=(('-- select a volume --', None), ('Volu…

Output(layout=Layout(border_bottom='1px solid #ccc', border_left='1px solid #ccc', border_right='1px solid #cc…